## code to translate German to English and vice versa with deepL

In [2]:
import pandas as pd
from pprint import pprint
import os
import ast
import deepl
from tqdm import tqdm
import json
import requests
import matplotlib.pyplot as plt

In [3]:
# get connection to deepl api
deepl_key = open("/Users/sofiaromero/Documents/DeepL.txt").read()
deepl_client = deepl.DeepLClient(deepl_key)

In [4]:
# api functions 

def generate_glossaries(dict_path_l, s_lang_l, t_lang_l, auth_key):
    url = "https://api.deepl.com/v3/glossaries"
    header= {
        "Authorization": f"DeepL-Auth-Key {auth_key}",
        "User-Agent": "YourApp/1.2.3",
        "Content-Type": "application/json"
    }    
    glossaries = []
    for i,d in enumerate(dict_path_l):
        payload = {
            "name": f"glossary{i}",
            "dictionaries": [{
                "source_lang": s_lang_l[i],
                "target_lang": t_lang_l[i],
                "entries": open(d,"r").read(),
                "entries_format": "csv"}
            ]}
        resp = requests.post(url, headers=header, json=payload)
        print(resp.status_code, resp.json())
        glossaries.append(resp.json())
    return glossaries


def get_glossary_ids(auth_key):
    url = "https://api.deepl.com/v3/glossaries"
    headers = {
        "Authorization": f"DeepL-Auth-Key {auth_key}",
        "User-Agent": "YourApp/1.2.3",
    }
    resp = requests.get(url, headers=headers, timeout=30)
    print(resp.status_code)
    return [x.get("glossary_id") for x in resp.json()["glossaries"]]


def transalate_to(text_l, s_lang, t_lang, glossary_id):
    res = []
    for text in text_l:
        if glossary_id:
            res.append(deepl_client.translate_text(text, target_lang=t_lang, source_lang=s_lang, glossary=glossary_id).text)
        else: 
            res.append(deepl_client.translate_text(text, target_lang=t_lang, source_lang=s_lang).text)
    return res


In [5]:
generate_glossaries(["config_files/ger_to_eng.csv", "config_files/eng_to_ger.csv"], ["DE","EN"], ["EN", "DE"], deepl_key)
# get all current glossaries
glossary_ids = get_glossary_ids(deepl_key)

201 {'glossary_id': '6d3581fe-81bf-4001-a4d9-51f88d6aeb1d', 'name': 'glossary0', 'dictionaries': [{'source_lang': 'de', 'target_lang': 'en', 'entry_count': 5}], 'creation_time': '2026-02-03T09:51:59.999524Z'}
400 {'message': 'Bad request. Reason: Invalid entries'}
200


In [6]:
# translate ki terms as test
'''
test_terms = list(ast.literal_eval(open("config_files/ger_anthroscore_KI_terms.txt").read()).values())
print(test_terms)
translations = transalate_to(test_terms, "DE", "EN-GB", glossary_ids[0])
print(translations)
'''

'\ntest_terms = list(ast.literal_eval(open("config_files/ger_anthroscore_KI_terms.txt").read()).values())\nprint(test_terms)\ntranslations = transalate_to(test_terms, "DE", "EN-GB", glossary_ids[0])\nprint(translations)\n'

In [7]:
# write text (from file) and its transaltion to df and save to csv and json file
def tranlate_to_df(txts_json, file_to):
    # load json as dict
    with open(txts_json, "r") as f:
        txts = json.load(f)
    if list(txts.values())[0].__class__ == str:  # if list of strings
        res_df = pd.DataFrame([(key, txts[key]) for key in txts.keys()], columns=['id', 'text']).drop_duplicates(subset=['text']).reset_index(drop=True)
    else:
        res_df = pd.DataFrame([(key, var) for (key, L) in txts.items() for var in L], columns=['id', 'text']).drop_duplicates(subset=['text']).reset_index(drop=True)
    #fill df with traslations
    res_df['translation'] = transalate_to(res_df['text'].tolist(), "DE", "EN-GB", glossary_ids[0])
    #save to file 
    res_df.to_csv(f"{file_to}.csv", index=False)
    res_df.to_json(f"{file_to}.json", orient="records")
    return res_df


### Translate KI_sentences to English

In [8]:
#tranlate_to_df("generated_data/KI_sentences.json", "generated_data/KI_sentences_translated")

In [9]:
#tranlate_to_df("generated_data/KI_sentences_full.json", "generated_data/KI_sentences_full_translated")

In [19]:
# getLeipzig 2024_1M data to translate
with open("generated_data/Leipzig_KI_sentences_all.json", "r") as f:
    leipzig_full = json.load(f)

leipzig24 = leipzig_full['2024_1M']
with open('generated_data/Leipzig_KI_sentences_24_1M.json', 'w') as f:
    json.dump(leipzig24, f)


In [ ]:
tranlate_to_df("generated_data/Leipzig_KI_sentences_24_1M.json", "generated_data/Leipzig_KI_sentences_24_1M_translated")

,id,text,translation
0,41,aber das was der maturand in dem erwähnten tex...,but what the high school graduate did in the t...
1,153,ai vorzustellen ein beweis für unsere bemühung...,introducing ai is a testament to our efforts i...
2,235,allerdings wirft der einsatz der großen sprach...,"however, the use of the large language models ..."
3,721,auch sollen die akkus im ai pin selbst sowie i...,"also, the batteries in the ai pin itself and i..."
4,761,auf dem mwc 2024 in barcelona kündigte das unt...,"at mwc 2024 in barcelona this week, the compan..."
5,1234,bislang kann man openai nicht einholen obwohl ...,so far you can't catch up with openai although...
6,1600,darüber hinaus haben künstliche intelligenz ki...,"in addition, artificial intelligence ki and ma..."
7,2016,das sprachmodell neigt dazu vermeintliche fakt...,the language model tends to invent supposed fa...
8,2133,datenschutz steht im zentrum der diskussion we...,data protection is at the centre of the discus...
9,2310,denn wenn ihr gemini herunterladet und install...,"because when you download and install Gemini, ..."


In [19]:
# input to anthroscore
# get AI-related words, sorted according to 'Häufigkeitsklassen' and terms from AnthroScore
# https://corpora.uni-leipzig.de/de/res?corpusId=deu_news_2024&word=KI
ki_terms = list(ast.literal_eval(open("config_files/ger_anthroscore_KI_terms.txt").read()).values())
ki_terms += ['KI','Künstliche Intelligenz', 'ChatGPT', 'AI', 'Copilot', 'Gemini', 'Chatbot','LLM', 'Chat-GPT',
            'Sprachmodell', 'KI-Modell', 'KI-System', 'KI-Tool', 'Artificial Intelligence', 'KI-Anwendung']
ai_terms = transalate_to(ki_terms, "DE", "EN-GB", glossary_ids[0])

In [20]:
with open(f"config_files/ai_terms.txt", "w") as f:
        f.writelines([" "+f"'{x}'" for x in ai_terms])
        f.close()

with open(f"config_files/ki_terms.txt", "w") as f:
        f.writelines([" "+f"'{x}'" for x in ki_terms])
        f.close()